# 실습 - 객체 탐지

**혈액세포의 종류와 위치 검출**

**작업 개요**

- 작업 이름 — `object-detection`
- 입력 — 이미지 한 장
- 출력 — 라벨(label) · 점수(score) · 위치 상자(box)
- 대상마다 하나씩 출력 — 개수 파악 가능
- 좌표 기준 — 이미지 좌측 상단이 (0, 0)

**분류와의 차이**

| 구분 | 이미지 분류 | 객체 탐지 |
|---|---|---|
| 출력 개수 | 이미지당 1개 | 대상마다 1개 |
| 출력 항목 | 라벨 · 점수 | 라벨 · 점수 · **위치** |
| 활용 | 전체 판별 | 개수 계산 · 위치 확인 |

**데이터**

- 현미경으로 촬영한 혈액 도말 이미지
- 범주 3종 — `rbc`(적혈구) · `wbc`(백혈구) · `platelets`(혈소판)
- 세포마다 위치 상자가 정답으로 포함

**모델**

- 동일 데이터로 학습된 DETR 탐지 모델

**실습 순서**

- 환경 준비
- 데이터 불러오기 · 구조 확인
- 모델 준비 · 추론 실행
- 결과 읽기 · 범주별 개수 집계
- 예측 상자 시각화 · 정답과 대조

---
## 0. 환경 준비

- `datasets` — Hub 데이터셋 내려받기
- `transformers` — 사전학습 모델로 추론 수행
- `PIL` — 이미지에 상자 그리기

In [ ]:
from datasets import load_dataset
from transformers import pipeline
from PIL import ImageDraw
from collections import Counter
import torch

print("준비 완료")
print("GPU 사용 가능:", torch.cuda.is_available())

---
## 1. 데이터 준비

### 1-1. 데이터셋 불러오기

- `name="full"` — 데이터셋의 설정(config) 지정
- 설정이 여러 개인 데이터셋은 반드시 지정해야 함
- `split="train[:100]"` — 앞 100장만 사용

In [ ]:
ds = load_dataset(
    "keremberke/blood-cell-object-detection",
    name="full",
    split="train[:100]"
)

ds

### 1-2. 항목 구조 확인

- 탐지 데이터는 이미지와 **위치 정보**를 함께 저장
- `objects` 열에 좌표(`bbox`)와 범주(`category`)가 담김
- 열 이름은 데이터셋마다 상이 — 확인 후 사용

In [ ]:
print("열:", ds.column_names)
print()

item = ds[0]
for k, v in item.items():
    print(f"{k}: {str(v)[:200]}")

### 1-3. 이미지와 정답 확인

- 한 장에 세포 수십 개가 포함
- `objects["bbox"]` — 세포마다 하나의 좌표
- `objects["category"]` — 세포마다 하나의 범주 번호

In [ ]:
image = ds[0]["image"]

print("이미지 크기:", image.size)
print("정답 개수:", len(ds[0]["objects"]["bbox"]))
print("첫 좌표:", ds[0]["objects"]["bbox"][0])
print("첫 범주:", ds[0]["objects"]["category"][0])

image

---
## 2. 모델 준비

- `task` — 수행할 작업 이름
- `model` — 혈액세포 데이터로 학습된 모델
- `device=0` — GPU에서 실행

첫 실행 시 모델 내려받기로 시간이 소요된다.

In [ ]:
det = pipeline(
    task="object-detection",
    model="theodullin/detr-resnet-50_finetuned_blood_cell_15epochs",
    device=0
)

print("모델 범주:", det.model.config.id2label)

---
## 3. 추론 실행

- `threshold` — 기준값 미만의 예측은 제외
- 값이 낮을수록 검출 수 증가 · 오탐도 증가
- 검출 수가 많으므로 개수를 먼저 확인

In [ ]:
result = det(image, threshold=0.5)

print("검출 수:", len(result))
result[:3]

### 3-1. 결과 읽기

- **label · score** — 이미지 분류와 동일한 형식
- **box** — 새로 추가된 항목, 대상의 위치
- 좌표 형식 — `xmin` · `ymin` · `xmax` · `ymax`

In [ ]:
r = result[0]

print("라벨:", r["label"])
print("점수:", round(r["score"], 4))
print("위치:", r["box"])

### 3-2. 범주별 개수 집계

- 탐지의 실질적 활용 — **자동 계수**
- 혈액 검사에서 세포 종류별 수를 세는 작업에 해당

In [ ]:
counts = Counter(r["label"] for r in result)

for label, n in counts.most_common():
    print(label, "-", n, "개")

### 3-3. 기준값 비교

- `threshold` 를 바꾸어 검출 수 변화 확인
- 낮추면 놓치는 세포는 줄지만 오탐이 늘어남

In [ ]:
for th in [0.3, 0.5, 0.7, 0.9]:
    n = len(det(image, threshold=th))
    print(f"threshold {th} — 검출 {n}개")

print("정답 개수:", len(ds[0]["objects"]["bbox"]))

---
## 4. 결과 시각화

### 4-1. 예측 상자 그리기

- `image.copy()` — 원본 보존을 위한 사본 생성
- `ImageDraw.Draw` — 이미지에 도형을 그리는 도구
- 코드는 결과 확인용 — 이해보다 실행에 중점

In [ ]:
img = image.copy().convert("RGB")
draw = ImageDraw.Draw(img)

for r in result:
    b = r["box"]
    draw.rectangle(
        (b["xmin"], b["ymin"], b["xmax"], b["ymax"]),
        outline="red", width=2
    )

img

### 4-2. 정답과 대조

- **초록** — 데이터에 포함된 정답 위치
- **빨강** — 모델이 예측한 위치
- 두 상자가 겹칠수록 정확한 예측

**좌표 형식 차이에 주의**

- 정답 — `[x, y, 너비, 높이]`
- 예측 — `xmin` · `ymin` · `xmax` · `ymax`
- 정답을 그릴 때는 `x + 너비`, `y + 높이` 로 변환

In [ ]:
img = image.copy().convert("RGB")
draw = ImageDraw.Draw(img)

# 정답 — 초록
for box in ds[0]["objects"]["bbox"]:
    x, y, w, h = box
    draw.rectangle((x, y, x + w, y + h), outline="lime", width=2)

# 예측 — 빨강
for r in result:
    b = r["box"]
    draw.rectangle(
        (b["xmin"], b["ymin"], b["xmax"], b["ymax"]),
        outline="red", width=2
    )

img

**확인 사항**

- 겹치는 상자 — 정확히 검출
- 초록만 있는 위치 — 놓친 세포
- 빨강만 있는 위치 — 잘못 검출

수치 평가에는 별도 지표(mAP 등)가 필요하며, 본 실습 범위를 벗어난다.

---
## 5. 다른 사례 확인

- 이미지 번호를 바꾸어 다른 사례 확인
- `threshold` 를 조절하여 결과 변화 관찰

In [ ]:
n = 10
img2 = ds[n]["image"]

res2 = det(img2, threshold=0.5)
print("검출:", len(res2), "/ 정답:", len(ds[n]["objects"]["bbox"]))
print(Counter(r["label"] for r in res2))

draw2 = ImageDraw.Draw(img2 := img2.copy().convert("RGB"))
for r in res2:
    b = r["box"]
    draw2.rectangle((b["xmin"], b["ymin"], b["xmax"], b["ymax"]),
                    outline="red", width=2)
img2

---
## 6. 정리

**수행 내용**

- 혈액세포 데이터로 학습된 모델을 그대로 사용 (추론)
- 이미지 한 장 → 세포별 라벨 · 점수 · 위치 산출
- 범주별 개수 집계 · 정답과 시각적 대조

**분류와의 차이**

- 출력에 **위치(box)** 가 추가됨
- 대상마다 결과가 나오므로 **개수 계산** 가능
- 코드 구조는 동일 — 작업 이름과 모델만 변경

**주의**

- `threshold` 설정이 검출 수를 좌우
- 정답과 예측의 좌표 형식이 상이 — 변환 필요
- 학습 범주 밖의 대상은 검출 불가

---

데이터 출처: keremberke/blood-cell-object-detection (Hugging Face Hub 공개)